# Libraries

In [0]:
%pip install numpy==1.26 pymc-marketing==0.15 openpyxl
%restart_python

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import arviz as az
import mlflow

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error as mse, mean_absolute_percentage_error as mape
from pymc_marketing.mmm import MMM, GeometricAdstock, LogisticSaturation
from pymc_marketing.prior import Prior
from scipy.optimize import curve_fit

mlflow.autolog(disable=True)

# Data

## Reading

In [0]:
directory = '/Volumes/training_feature_development/marketing_mix_modelling/data_v1/model_data/'

model_df = pd.read_csv(directory + 'lbe_mmm.csv')

model_df['week'] = pd.to_datetime(model_df['week'])

# Linear Regression

In [0]:
X = model_df[model_df.columns[1:-2]]
y = model_df['revenue']

In [0]:
model = LinearRegression().fit(X, y)
y_pred = model.predict(X)

In [0]:
plt.figure(figsize=(8, 6), dpi=120)
plt.plot(y, color='grey', alpha=0.5, label='Actual')
plt.plot(y_pred, color='#065CB1', label='Predicted')
plt.gca().yaxis.set_major_formatter(mtick.EngFormatter())
plt.ylim(0, 5*10**6)
plt.xlabel('Week')
plt.ylabel('Revenue ($)')
plt.legend(loc='upper left')
plt.grid(axis='y', alpha=0.25)
plt.show()

print(f'R2: {model.score(X, y):.3f}')
print(f'MAPE: {mape(y, y_pred):.3f}')

# Marketing Mix Model

In [0]:
X = model_df[model_df.columns[:-2]]
y = model_df['revenue']

In [0]:
# Setting assumptions
custom_config = {
    # Non-negative baseline sales
    'intercept': Prior('Exponential', lam=2),
    # Setting a non-negative control effect
    'gamma_control': Prior('HalfNormal', sigma=2),
    # Setting an average retention rate of 60% 
    'adstock_alpha': Prior('Beta', alpha=3, beta=2),
    # Setting a low ceiling for spend
    'saturation_lam': Prior('Gamma', alpha=5, beta=1)
}

mmm = MMM(model_config=custom_config,
          date_column='week',
          channel_columns=X.columns[4:],
          control_columns=X.columns[1:4],
          # Maximum effect over 4 weeks
          adstock=GeometricAdstock(l_max=4),
          saturation=LogisticSaturation())

mmm.fit(X, y, chains=2, random_seed=12, target_accept=0.95)

mmm.sample_prior_predictive(X, extend_idata=True, combined=True)
mmm.sample_posterior_predictive(X, extend_idata=True, combined=True)

y_pred = mmm.predict(X)

In [0]:
plt.figure(figsize=(8, 6), dpi=120)

# Plotting model
plt.plot(X['week'], y_pred, color='#065CB1', label='Predicted')
plt.plot(X['week'], y, color='grey', alpha=0.5, label='Actual')

# Changing aesthetics
plt.ylim(0, 5*10**6)
plt.xlabel('Week')
plt.ylabel('Revenue ($)')
plt.gca().yaxis.set_major_formatter(mtick.EngFormatter(unit=''))
plt.grid(axis='y', alpha=0.25)
plt.legend(loc='upper left')

plt.show()

In [0]:
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mse(y, y_pred))
mape = mape(y, y_pred)

print(f'R2: {r2:.3f}')
print(f'RMSE: {rmse:.0f}')
print(f'MAPE: {mape:.3f}')

In [0]:
mmm.plot_waterfall_components_decomposition();

In [0]:
f"Total spend (trailing 2 years): ${model_df[mmm.channel_columns].select_dtypes('number').sum().sum():,.0f}"

In [0]:
fig = mmm.plot_direct_contribution_curves(same_axes=True, show_fit=True, quantile_lower=0.5, quantile_upper=0.5, xlim_max=1.5)
fig.set_size_inches(12, 8)
ax = fig.gca()

# Changing aesthetics
ax.xaxis.set_major_formatter(mtick.EngFormatter(unit=''))
ax.yaxis.set_major_formatter(mtick.EngFormatter(unit=''))
ax.grid(alpha=0.25)
ax.legend(loc='center left', bbox_to_anchor=(0.78, 0.74))

plt.show()

In [0]:
fig = mmm.plot_channel_contribution_grid(start=0.5, stop=2, num=16)

for collection in fig.gca().collections:
    collection.remove()
for line in fig.gca().get_lines():
    if line.get_linestyle() == '--' and line.get_visible():
        line.set_alpha(0.3)

fig.set_size_inches(12, 8)

# Changing aesthetics
fig.gca().yaxis.set_major_formatter(mtick.EngFormatter(unit=''))
fig.gca().grid(alpha=0.25)
fig.gca().set_ylim(0, 4*10**7)
plt.show()

In [0]:
channel_contribution_original_scale = mmm.compute_channel_contribution_original_scale()
spend_sum = X[mmm.channel_columns].sum().to_numpy()

roas_samples = (
    channel_contribution_original_scale.sum(dim="date")
    / spend_sum[np.newaxis, np.newaxis, :]
)

mean_roas = roas_samples.mean(dim='chain').mean(dim='draw').values.round(1)
roas_df = pd.DataFrame({
    'channel': mmm.channel_columns,
    'avg_roas': mean_roas
})
roas_df = roas_df.sort_values(by='avg_roas').reset_index(drop=True)

plt.figure(figsize=(8, 6), dpi=240)
plt.barh(roas_df['channel'], roas_df['avg_roas'], color='#065CB1', zorder=2)

# Changing aesthetics
plt.gca().set_xlabel('Average ROAS')
plt.gca().set_ylabel('Channel')
plt.gca().grid(axis='x', alpha=0.25)

plt.show()

# Marketing Mix Model (Aggregated)

In [0]:
X = model_df[['week', 'summer', 'holiday', 'week_num']]
X = X.merge(model_df[model_df.columns[4:-3]].sum(axis=1).rename('spend'), how="left", left_index=True, right_index=True)
X = X[X['week'].between('2024-06-28', '2025-06-27')]
X['spend'] = X['spend'] * 1.5
y = model_df[model_df['week'].between('2024-06-28', '2025-06-27')]['revenue']

In [0]:
# Setting assumptions
custom_config = {
    # Non-negative baseline sales
    'intercept': Prior('Exponential', lam=2),
    # Setting a non-negative control effect
    'gamma_control': Prior('HalfNormal', sigma=2),
    # Setting an average retention rate of 60% 
    'adstock_alpha': Prior('Beta', alpha=3, beta=2),
    # Setting a low ceiling for spend
    'saturation_lam': Prior('Gamma', alpha=6, beta=1)
}

mmm = MMM(model_config=custom_config,
          date_column='week',
          channel_columns=X.columns[[4]],
          control_columns=X.columns[1:4],
          # Maximum effect over 4 weeks
          adstock=GeometricAdstock(l_max=4),
          saturation=LogisticSaturation())

mmm.fit(X, y, chains=2, random_seed=12, target_accept=0.95)

mmm.sample_prior_predictive(X, extend_idata=True, combined=True)
mmm.sample_posterior_predictive(X, extend_idata=True, combined=True)

y_pred = mmm.predict(X)

In [0]:
fig = mmm.plot_channel_contribution_grid(start=0.5, stop=5, num=29, absolute_xrange=True)
for collection in fig.gca().collections:
    collection.remove()
fig.set_size_inches(12, 8)
fig.set_dpi(120)

# Changing aesthetics
fig.gca().xaxis.set_major_formatter(mtick.EngFormatter(unit=''))
fig.gca().yaxis.set_major_formatter(mtick.EngFormatter(unit=''))
fig.gca().set_xticks(np.arange(0, 2.5*10**7, 2*10**6))
fig.gca().set_yticks(np.arange(0, 2.9*10**7, 2*10**6))
fig.gca().set_ylim(0, 2.9*10**7)
fig.gca().set_xlabel('Annual Budget ($)', fontsize=14)
fig.gca().set_ylabel('Annual Revenue Contribution ($)', fontsize=14)
plt.grid(alpha=0.25)
plt.legend(loc='upper right', bbox_to_anchor=(1, 0.99), fontsize=14)
fig.gca().tick_params(axis='both', which='major', labelsize=14)
fig.gca().title.set_fontsize(18)

plt.show()

In [0]:
x = X['spend'].reset_index(drop=True)*52
y = mmm.compute_mean_contributions_over_time(original_scale=True)['spend'].reset_index(drop=True)*52
z = y/x

def mod(x, a, b):
    return a + b * np.log(x)

params, _ = curve_fit(mod, x, y)
a, b = params

x_fit = np.linspace(3*10**6, 3.3*10**7, 15)
y_fit = mod(x_fit, a, b)
z_fit = np.gradient(y_fit, x_fit)

plt.figure(figsize=(12, 8), dpi=120)
plt.plot(x_fit, z_fit, marker='o', color='tab:blue')

# Changing aesthetics
plt.vlines(x=6*10**6, ymin=0, ymax=1.5, linestyle='--', color='black', label='2024 Budget', linewidth=1)
plt.xticks(np.arange(0, 3.4*10**7, 2*10**6))
plt.gca().xaxis.set_major_formatter(mtick.EngFormatter(unit=''))
plt.xlim(0, 3.4*10**7)
plt.ylim(0, 1.5)
plt.title('Marginal Contribution of Marketing Budget')
plt.xlabel('Annual Budget ($)')
plt.ylabel('Marginal Attendance Contribution')
plt.legend()
plt.grid(alpha=0.25)

plt.show()